# Chirplet Decomposition Spike

**Stage 3 of the pipeline.**
Spike, not production.

**Exit gate:** given an MP3/WAV from `pipeline/data/raw/`, produce `Vector{ChirpletAtom}` where each atom carries `(t, f, chirp_rate, duration, amplitude)`, and **resynthesis from atoms produces audibly recognizable song**.
Invertibility is the gate, not reconstruction error.

Goal of this notebook:
1. Pick one recording, look at its spectrogram.
2. Run a candidate chirplet decomposition.
3. Scatter atoms over the spectrogram.
4. Resynthesize from atoms and play it back.
5. Eyeball atom counts, time, memory per recording.

Out of scope: the spatial model (Stage 4), pipeline integration (after the spike), per-syllable segmentation (atoms are the unit).

In [ ]:
using Revise
using DSP
using FFTW
using WAV
using FileIO
using Plots
using BenchmarkTools
using Statistics
using LinearAlgebra
using Base64

## 1. Pick a Recording

WAV reads natively; MP3 needs `MP3.jl` or an `ffmpeg` shell-out.
Start with one of the WAVs to keep the spike honest.

In [ ]:
const RAW_DIR = joinpath(@__DIR__, "..", "data", "raw")
wav_files = filter(f -> endswith(lowercase(f), ".wav"), readdir(RAW_DIR; join=true))

In [ ]:
# Edit the index to switch recordings.
selected_file = wav_files[1]
signal, fs = wavread(selected_file)
x = vec(mean(signal; dims=2))   # mono
duration_s = length(x) / fs
(file=basename(selected_file), samples=length(x), fs=Int(fs), duration_s=round(duration_s; digits=2))

## 2. Spectrogram Baseline

Before chirplet decomposition, look at the song the obvious way.
This is the picture the atoms have to reproduce.

In [ ]:
spec = spectrogram(x, 1024, 512; fs=fs, window=hanning)
heatmap(time(spec), freq(spec), 10 .* log10.(power(spec) .+ 1e-12);
    xlabel="time (s)", ylabel="frequency (Hz)", colorbar_title="dB",
    title="spectrogram: $(basename(selected_file))")

## 3. The Chirplet Atom Data Structure

A Gaussian-windowed linear chirp:

$$g(t; t_0, f_0, c, \sigma, A) = A \cdot \exp\!\left(-\frac{(t-t_0)^2}{2\sigma^2}\right) \cdot \cos\!\bigl(2\pi (f_0 (t - t_0) + \tfrac{c}{2}(t - t_0)^2) \bigr)$$

Parameters: center time `t₀`, center freq `f₀`, chirp rate `c` (Hz/s), duration scale `σ`, amplitude `A`.

In [ ]:
struct ChirpletAtom
    t::Float64        # center time (s)
    f::Float64        # center frequency (Hz)
    c::Float64        # chirp rate (Hz/s)
    σ::Float64        # duration scale (s)
    A::Float64        # amplitude
end

function synthesize(atom::ChirpletAtom, t::AbstractVector)
    Δ = t .- atom.t
    envelope = exp.(-Δ.^2 ./ (2 * atom.σ^2))
    phase = 2π .* (atom.f .* Δ .+ 0.5 .* atom.c .* Δ.^2)
    atom.A .* envelope .* cos.(phase)
end

function synthesize(atoms::Vector{ChirpletAtom}, t::AbstractVector)
    out = zeros(length(t))
    for a in atoms
        out .+= synthesize(a, t)
    end
    out
end

## 4. Placeholder Decomposition

Replace this with a real algorithm.
Candidates:

- **Adaptive matching pursuit** over a chirplet dictionary (Mann & Haykin 1991).
- **MP / OMP** with a precomputed atom library (faster, less faithful).
- Roll your own in ~100 lines.

The placeholder below returns a hand-picked atom so resynthesis end-to-end runs even before the algorithm exists.
Delete once a real decomposer lands.

In [ ]:
function decompose_placeholder(x, fs; n_atoms=1)::Vector{ChirpletAtom}
    duration_s = length(x) / fs
    [ChirpletAtom(duration_s/2, 4000.0, 0.0, 0.05, 0.3) for _ in 1:n_atoms]
end

atoms = decompose_placeholder(x, fs; n_atoms=1)

## 5. Atoms over the Spectrogram

Each atom is a tilted line in the t-f plane: center `(t, f)`, slope `c`, extent `σ` along time.
Overlay these on the spectrogram to eyeball whether the decomposition is finding the right structure.

In [ ]:
function plot_atoms_over_spectrogram(spec, atoms)
    p = heatmap(time(spec), freq(spec), 10 .* log10.(power(spec) .+ 1e-12);
        xlabel="time (s)", ylabel="frequency (Hz)", alpha=0.7)
    for a in atoms
        ts = range(a.t - 2a.σ, a.t + 2a.σ; length=20)
        fs_atom = a.f .+ a.c .* (ts .- a.t)
        plot!(p, ts, fs_atom; lw=2, color=:white, label=nothing)
        scatter!(p, [a.t], [a.f]; color=:red, markersize=4, label=nothing)
    end
    p
end

plot_atoms_over_spectrogram(spec, atoms)

## 6. Resynthesize and Listen

**This is the exit gate for Stage 3.**
If the resynthesis is audibly recognizable as the original song (or as song-like structure from the same recording), the decomposition is good enough to move on.
Numeric reconstruction error is secondary: the spatial model only ever sees atoms, so the ear is the judge.

In [ ]:
t_axis = range(0, length(x)/fs; length=length(x))
x_recon = synthesize(atoms, t_axis)
x_recon_norm = x_recon ./ (maximum(abs, x_recon) + 1e-9);

In [ ]:
# Inline audio player. Embeds the WAV as base64, so it works in VS Code and Jupyter.
function play(samples, fs)
    io = IOBuffer()
    wavwrite(samples, io; Fs=Int(fs))
    b64 = base64encode(take!(io))
    display("text/html", """<audio controls src="data:audio/wav;base64,$b64"></audio>""")
end

play(x_recon_norm, fs)

In [ ]:
# Original, for A/B comparison.
play(x ./ (maximum(abs, x) + 1e-9), fs)

## 7. Cost: Atoms, Time, Memory

Stage 3 also needs answers for: how many atoms per recording, how long does decomposition take per recording, how much memory.
These numbers feed into Stage 4 (spatial fit) and Stage 5 (Cloud Run sizing).

In [ ]:
@benchmark decompose_placeholder($x, $fs; n_atoms=10)

## Notes and Next Steps

- [ ] Replace `decompose_placeholder` with a real decomposer.
- [ ] Decide MP3 read path (`MP3.jl` vs `ffmpeg` shell-out); currently WAV-only.
- [ ] Once invertibility holds, lift the working function into `pipeline/src/dsp/` and let Revise pick it up.
- [ ] Then write a workflow in `pipeline/src/workflows/` that decomposes every row in `recordings`.